In [1]:
pip install pandas google-generativeai matplotlib

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime

GOOGLE_SHEET_CSV_URL = "https://docs.google.com/spreadsheets/d/1c5K4f49a_HLaRel0xp-sah9MAEyqpSz8zg4KmHCKjqs/export?format=csv&gid=428263152"


def _normalize_date_label(date_label):
    label = str(date_label).strip()
    parsed = pd.to_datetime(
        f"{label} {datetime.now().year}",
        format="%b %d %Y",
        errors="coerce",
    )
    if pd.isna(parsed):
        parsed = pd.to_datetime(label, errors="coerce")
    return label, parsed


def _format_date_label(ts):
    return f"{ts.strftime('%b')} {ts.day}"


def _extract_unidil_rows(raw_df):
    first_col = raw_df.iloc[:, 0].astype(str).str.strip().str.upper()
    unidil_rows = first_col[first_col == "UNIDIL"]

    if unidil_rows.empty:
        return [raw_df]

    blocks = []
    start_indexes = list(unidil_rows.index)
    for idx, start_idx in enumerate(start_indexes):
        end_idx = start_indexes[idx + 1] if idx + 1 < len(start_indexes) else len(raw_df)
        block = raw_df.iloc[start_idx:end_idx].reset_index(drop=True)
        blocks.append(block)
    return blocks


def _parse_production_sheet(raw_df):
    if raw_df.shape[0] < 4:
        return pd.DataFrame(columns=["Date", "Process", "Metric", "Value"])

    date_row = raw_df.iloc[2]
    data = []
    current_section = None

    for i in range(3, len(raw_df)):
        label = str(raw_df.iloc[i, 0]).strip().lower()

        if "planned" in label:
            current_section = "Planned"
            continue
        if "actual" in label:
            current_section = "Actual"
            continue
        if "yield" in label:
            current_section = "Yield"
            continue
        if "rejection" in label:
            current_section = "Rejections"
            continue
        if "stoppage" in label:
            current_section = "Stoppages"
            continue

        if not current_section or pd.isna(raw_df.iloc[i, 0]):
            continue

        process = None
        if "corrugator" in label:
            process = "Corrugator"
        elif "tuber" in label:
            process = "Tuber"

        if not process:
            continue

        for col in range(1, len(raw_df.columns)):
            val = raw_df.iloc[i, col]
            date_val = date_row[col]

            if pd.isna(val) or pd.isna(date_val):
                continue

            val_str = str(val).strip()
            if val_str in ["-", "", "nan"]:
                continue

            clean_val = val_str.replace(",", "").replace("..", ".")
            try:
                numeric_val = float(clean_val)
            except ValueError:
                continue

            data.append(
                {
                    "Date": str(date_val).strip(),
                    "Process": process,
                    "Metric": current_section,
                    "Value": numeric_val,
                }
            )

    return pd.DataFrame(data)


def load_unidil_data(filepath, google_sheet_url=GOOGLE_SHEET_CSV_URL):
    local_raw_df = pd.read_csv(filepath, header=None)
    local_long_df = _parse_production_sheet(local_raw_df)

    google_long_df = pd.DataFrame(columns=["Date", "Process", "Metric", "Value"])
    try:
        google_raw_df = pd.read_csv(google_sheet_url, header=None)
        unidil_blocks = _extract_unidil_rows(google_raw_df)
        parsed_blocks = [_parse_production_sheet(block) for block in unidil_blocks]
        google_long_df = pd.concat(parsed_blocks, ignore_index=True)
    except Exception:
        pass

    clean_df = pd.concat([local_long_df, google_long_df], ignore_index=True)
    clean_df = clean_df.drop_duplicates(subset=["Date", "Process", "Metric"], keep="last")

    final_df = clean_df.pivot_table(
        index=["Date", "Process"], columns="Metric", values="Value", aggfunc="first"
    ).reset_index()
    final_df.columns.name = None

    for col in ["Planned", "Actual", "Yield", "Rejections", "Stoppages"]:
        if col not in final_df.columns:
            final_df[col] = 0
        else:
            final_df[col] = final_df[col].fillna(0)

    normalized = final_df["Date"].apply(_normalize_date_label)
    final_df["Date_Parsed"] = [item[1] for item in normalized]
    final_df = final_df.dropna(subset=["Date_Parsed"]).copy()

    all_dates = pd.date_range(final_df["Date_Parsed"].min(), final_df["Date_Parsed"].max(), freq="D")
    all_processes = sorted(final_df["Process"].dropna().unique().tolist())
    calendar_df = pd.MultiIndex.from_product([all_dates, all_processes], names=["Date_Parsed", "Process"]).to_frame(index=False)

    final_df = calendar_df.merge(
        final_df.drop(columns=["Date"]),
        on=["Date_Parsed", "Process"],
        how="left",
    )

    for col in ["Planned", "Actual", "Yield", "Rejections", "Stoppages"]:
        final_df[col] = final_df[col].fillna(0)

    final_df["Date"] = final_df["Date_Parsed"].apply(_format_date_label)
    final_df = final_df.sort_values(by=["Date_Parsed", "Process"]).reset_index(drop=True)

    return final_df



In [ ]:
df = load_unidil_data("Daily Production Data_NEW Format - UNIDIL.csv")
print(df.head())



In [ ]:
import matplotlib.pyplot as plt

corr_df = df[df["Process"] == "Corrugator"].sort_values("Date_Parsed")
tuber_df = df[df["Process"] == "Tuber"].sort_values("Date_Parsed")

plt.figure(figsize=(14, 5))
plt.plot(corr_df["Date_Parsed"], corr_df["Actual"], label="Corrugator Actual", linewidth=2)
plt.plot(corr_df["Date_Parsed"], corr_df["Planned"], label="Corrugator Planned", linewidth=2)
plt.title("Corrugator: Planned vs Actual (Daily)")
plt.xlabel("Date")
plt.ylabel("Production")
plt.legend()
plt.grid(alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

plt.figure(figsize=(14, 5))
plt.plot(tuber_df["Date_Parsed"], tuber_df["Actual"], label="Tuber Actual", linewidth=2)
plt.plot(tuber_df["Date_Parsed"], tuber_df["Planned"], label="Tuber Planned", linewidth=2)
plt.title("Tuber: Planned vs Actual (Daily)")
plt.xlabel("Date")
plt.ylabel("Production")
plt.legend()
plt.grid(alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()



In [4]:
def add_kpis(df):
    # Avoid division by zero
    df["Efficiency"] = np.where(
        df["Planned"] > 0, 
        (df["Actual"] / df["Planned"]) * 100, 
        0
    )
    df["Production_Loss"] = df["Planned"] - df["Actual"]
    
    # Ensure no negative loss if they over-produced
    df["Production_Loss"] = df["Production_Loss"].clip(lower=0)
    return df

def analyze_factory(df):
    insights = []
    
    for _, row in df.iterrows():
        process = row["Process"]
        date = row["Date"]
        
        if row["Efficiency"] > 0 and row["Efficiency"] < 90:
            insights.append(f"[{date}] {process}: Low efficiency ({row['Efficiency']:.1f}%).")
            
        if row["Production_Loss"] > 0:
            if row["Stoppages"] > 60:
                insights.append(f"[{date}] {process}: Missed target by {row['Production_Loss']:.1f} units. High downtime detected ({row['Stoppages']} mins).")
            elif row["Rejections"] > 50:
                insights.append(f"[{date}] {process}: Missed target by {row['Production_Loss']:.1f} units. High material rejection.")
                
    return insights

df = add_kpis(df)
insights = analyze_factory(df)
print(f"Generated {len(insights)} insights!")

Generated 107 insights!


In [ ]:
import google.generativeai as genai
import os

# For Streamlit Cloud, get the API key from environment variables (secrets)
GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY")
genai.configure(api_key=GOOGLE_API_KEY)
model = genai.GenerativeModel('gemini-2.5-flash')

def ai_agent(insights_list):
    # Only send the top 10 most recent/severe insights to save context
    summary_insights = "\n".join(insights_list[:10])
    
    prompt = f"""
    You are a manufacturing expert consulting for the Unidil factory.
    Here are the latest operational insights:
    
    {summary_insights}
    
    Provide a concise report including:
    1. Root causes for the production drops.
    2. Key bottlenecks in the Corrugator and Tuber processes.
    3. Practical, actionable improvement steps.
    """
    
    response = model.generate_content(prompt)
    return response.text

c:\Users\arush\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\arush\AppData\Local\Temp\ipykernel_34260\2458414526.py:1: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


Here's a concise report on the Tuber line's operational insights:

**Unidil Factory - Tuber Process Performance Report (Feb 10-16)**

**1. Root Causes for Production Drops:**
The consistent production drops on the Tuber line are primarily driven by two interlinked issues:
*   **Excessive Downtime:** The machine is frequently non-operational, experiencing significant periods of downtime (ranging from 184 to 744 minutes daily) which directly leads to missed production targets.
*   **Suboptimal Operational Efficiency:** Even when running, the Tuber's efficiency is frequently low (55.9% to 88.3%), indicating underperformance, slower output rates, and potential for minor stops or speed losses.

**2. Key Bottlenecks:**
*   **Tuber Process:** The Tuber machine is the critical bottleneck, severely limiting the factory's overall output due to its poor availability (high downtime) and reduced performance (low efficiency).
*   **Corrugator Process:** No data was provided for the Corrugator proces

In [7]:
pip install tabulate

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import google.generativeai as genai
from IPython.display import display, Markdown

# 1. Prepare the Factory Context
# We convert the last 14 days of data to a Markdown table so the AI can read it easily
context_data = df.tail(14).to_markdown(index=False) 
context_insights = "\n".join(insights)

# 2. Define the AI's Persona and Knowledge
system_prompt = f"""
You are an expert industrial AI assistant for the Unidil factory.
Your goal is to answer questions about the factory's production performance accurately and concisely.

Here is the most recent production data (Last 14 Days):
{context_data}

Here are the automated KPI insights detected by the system:
{context_insights}

When asked a question, look at the data provided above. If the data doesn't contain the answer, politely say you don't have that information.
"""

# 3. Initialize the Model with the System Prompt
# Note: Ensure you have run genai.configure(api_key="...") in the cell above
chat_model = genai.GenerativeModel(
    model_name='gemini-2.5-flash',
    system_instruction=system_prompt
)

# Start a chat session (this automatically handles conversation history)
chat_session = chat_model.start_chat(history=[])

def unidil_chat():
    display(Markdown("### 🤖 Unidil AI Assistant Initialized"))
    print("Type your questions below. Type 'exit' or 'quit' to stop the chat.\n" + "-"*50)
    
    while True:
        # Get user input from the notebook cell
        user_input = input("👤 You: ")
        
        # Check for exit commands
        if user_input.lower() in ['exit', 'quit']:
            print("\n🤖 Session ended. Closing the chat.")
            break
            
        # Skip empty inputs
        if not user_input.strip():
            continue
            
        try:
            # Send the message to the AI
            response = chat_session.send_message(user_input)
            
            # Display the AI's response using Markdown for clean formatting
            print("\n")
            display(Markdown(f"**🤖 AI:** {response.text}"))
            print("-" * 50)
            
        except Exception as e:
            print(f"\n⚠️ An error occurred: {e}")

# 4. Run the chat loop
unidil_chat()

### 🤖 Unidil AI Assistant Initialized

Type your questions below. Type 'exit' or 'quit' to stop the chat.
--------------------------------------------------
